# Plant Disease Detection Using ResNet50

SE4050 – Deep Learning | Supervised multiclass plant-leaf disease classification

This notebook presents the completed ResNet50 experiment: dataset exploration, reproducible split/preprocessing, ImageNet transfer learning, fine-tuning, and final evaluation. It loads saved artifacts and does not retrain or rerun test evaluation unless explicitly enabled.

In [ ]:
from pathlib import Path
import csv, json, os, platform, random, sys
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from IPython.display import display, Image as DisplayImage

# Change only this path when sharing the notebook.
PROJECT_ROOT = Path('/Users/thakkifilyas/Desktop/DL_Project/plant-disease-repo')
DATA_DIR = Path('/Users/thakkifilyas/Desktop/DL_Project/PlantVillage')
RESULTS_DIR = PROJECT_ROOT / 'results' / 'resnet50'
EDA_DIR = RESULTS_DIR / 'eda'; EVALUATION_DIR = RESULTS_DIR / 'evaluation'
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'resnet50_config.yaml'
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Python:', platform.python_version()); print('TensorFlow:', tf.__version__)
print('Project root:', PROJECT_ROOT); print('Dataset:', DATA_DIR); print('Results:', RESULTS_DIR); print('Configuration:', CONFIG_PATH)
print('Devices:', tf.config.list_physical_devices())

## Reproducibility
All experiments use random seed 42. The original PlantVillage images remain unchanged. The final test split is reserved for the final evaluation only.

In [ ]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED); tf.keras.utils.set_random_seed(RANDOM_SEED)
print('Random seed:', RANDOM_SEED)

## Dataset and exploratory data analysis
PlantVillage class-folder names represent disease or healthy categories. Counts below are calculated from the selected local dataset; no statistics are invented.

In [ ]:
extensions = {'.jpg','.jpeg','.png','.bmp'}
class_names = sorted(d.name for d in DATA_DIR.iterdir() if d.is_dir())
rows = []
for class_name in class_names:
    count = sum(p.is_file() and p.suffix.lower() in extensions for p in (DATA_DIR / class_name).rglob('*'))
    rows.append({'Class ID': len(rows), 'Class Name': class_name, 'Image Count': count})
counts = rows
print('Number of classes:', len(class_names), '| Total images:', sum(r['Image Count'] for r in rows))
display(rows)

In [ ]:
labels = [r['Class Name'] for r in rows]; values = [r['Image Count'] for r in rows]
fig, ax = plt.subplots(figsize=(13,6)); ax.bar(labels, values, color='#3b7a57'); ax.set_title('PlantVillage Class Distribution'); ax.set_ylabel('Number of images'); ax.tick_params(axis='x', rotation=75); fig.tight_layout(); plt.show()
summary = json.loads((EDA_DIR/'summary.json').read_text())
print('Largest class:', summary['largest_class']); print('Smallest class:', summary['smallest_class']); print('Class imbalance ratio:', summary['max_to_min_ratio'])

In [ ]:
quality = json.loads((EDA_DIR/'summary.json').read_text())
print('Integrity files checked:', quality['integrity_files_checked']); print('Corrupt files found:', quality['corrupt_files_found'])
dimensions = []
for row in csv.DictReader(open(EDA_DIR/'representative_dimensions.csv')): dimensions.append(row)
display(dimensions[:15])
display(DisplayImage(filename=str(EDA_DIR/'representative_samples.png')))

## Dataset split and preprocessing
The shared manifest should be supplied through `manifest` in the YAML configuration when the group finalizes it. The completed experiment used seed 42, 224×224 RGB input, and a 70/15/15 train/validation/test split. Training-only augmentation consists of horizontal flip, ±9° rotation, 10% zoom, 5% translation, and 10% contrast variation. ImageNet `ResNet50.preprocess_input` is applied inside the model; validation/test preprocessing is deterministic.

In [ ]:
# Show saved training history and split-independent artifact information.
history = json.loads((RESULTS_DIR/'history.json').read_text())
training_metrics = list(csv.DictReader(open(RESULTS_DIR/'training_metrics.csv')))
print('Recorded epochs:', len(training_metrics)); print('Training summary:', json.loads((RESULTS_DIR/'training_summary.json').read_text()))
for stage, values in history.items(): print(stage, 'epochs:', len(values.get('loss', [])))
# Saved curves are displayed below; they are generated from the actual training run.
display(DisplayImage(filename=str(EVALUATION_DIR/'accuracy_curve.png'))); display(DisplayImage(filename=str(EVALUATION_DIR/'loss_curve.png')))

## Why ResNet50 and transfer learning?
ResNet50 is a 50-layer convolutional architecture built from residual blocks. A skip connection learns `F(x) + x`, improving gradient flow in deep networks. ImageNet weights provide general visual features; Stage A freezes the backbone and trains the new classifier, while Stage B unfreezes `conv5_block1` and later layers using a smaller learning rate to adapt high-level features to plant disease.

## Implementation: dataset loading and deterministic split
This is the actual dataset API used by the ResNet50 pipeline. If a shared CSV manifest with `path,label,split` is supplied, its assignments are used exactly; otherwise the loader creates the reproducible seed-42 split for this standalone experiment.

In [ ]:
from resnet50.dataset import load_records, make_datasets
records = load_records(str(DATA_DIR), manifest=None)
train_ds, validation_ds, test_ds, selected_classes = make_datasets(records, seed=RANDOM_SEED, image_size=(224,224), batch_size=32, validation_split=0.15, test_split=0.15)
print('Selected classes:', len(selected_classes)); print('Class names:', selected_classes)
print('Dataset split created in memory; source images were not copied or modified.')

## Implementation: preprocessing and augmentation
Images are decoded as 3-channel RGB and resized to 224×224 in `dataset.py`. The model applies ImageNet preprocessing. Augmentation is defined inside the model and is active only when `training=True`; validation and test calls use `training=False`.

In [ ]:
from resnet50.model import build_resnet50
model_for_inspection = build_resnet50(len(selected_classes), image_size=(224,224), weights=None, freeze_backbone=True, augment=True)
print('Output neurons:', model_for_inspection.output_shape[-1])
print('Training augmentation layers:', [layer.name for layer in model_for_inspection.layers if 'augmentation' in layer.name])

## Optional implementation: run training from this notebook
The following cell calls the real two-stage training implementation directly. It is disabled by default because the completed checkpoint and metrics already exist. Set `RUN_TRAINING = True` only when an intentional new experiment is required.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    from resnet50.train import main as train_resnet50
    train_resnet50(str(CONFIG_PATH))
    print('Training completed; checkpoint saved under results/resnet50/.')
else:
    print('Training skipped: using the saved completed experiment artifacts.')

In [ ]:
from resnet50.model import build_resnet50
model = build_resnet50(len(class_names), image_size=(224,224), weights=None, freeze_backbone=True, augment=False)
print('Model:', model.name); print('Classes:', len(class_names)); print('Total parameters:', model.count_params()); print('Trainable parameters:', sum(np.prod(v.shape) for v in model.trainable_weights)); print('Non-trainable parameters:', sum(np.prod(v.shape) for v in model.non_trainable_weights))
model.summary()

## Final evaluation results
The following artifacts were produced after model development and one final pass over the untouched test split. No test metric was used for tuning.

In [ ]:
metrics = json.loads((EVALUATION_DIR/'final_metrics.json').read_text())
for key in ['test_accuracy','test_precision_weighted','test_recall_weighted','test_f1_weighted','test_roc_auc_ovr_weighted','total_parameters','trainable_parameters','checkpoint_size_bytes','inference_seconds_per_image','throughput_images_per_second']:
    print(f'{key}: {metrics.get(key)}')
print('Classification report:'); print((EVALUATION_DIR/'classification_report.txt').read_text())
display(DisplayImage(filename=str(EVALUATION_DIR/'confusion_matrix.png'))); display(DisplayImage(filename=str(EVALUATION_DIR/'per_class_performance.png')))

## Experimental design and training configuration
The completed run uses seed 42, 224×224 RGB inputs, batch size 32, Adam, sparse categorical cross-entropy, learning rates 1e-4 and 1e-5, validation-loss early stopping, and ReduceLROnPlateau. The same dataset and split must be used by all group architectures for a fair comparison.

In [ ]:
config_rows = [{'Parameter':'Random seed','Value':42},{'Parameter':'Input size','Value':'224 × 224 RGB'},{'Parameter':'Batch size','Value':32},{'Parameter':'Stage A learning rate','Value':'1e-4'},{'Parameter':'Stage B learning rate','Value':'1e-5'},{'Parameter':'Optimizer','Value':'Adam'},{'Parameter':'Loss','Value':'Sparse categorical cross-entropy'},{'Parameter':'Fine-tuned layers','Value':'conv5_block1 and later'}]
display(config_rows)

## Training results
Training is not repeated here. The epoch table below is loaded from the saved run and includes the available learning rate and validation metrics.

In [ ]:
display(training_metrics)
best_index = int(np.argmax([float(r['val_accuracy']) for r in training_metrics])); final_row = training_metrics[-1]
print('Best validation accuracy:', training_metrics[best_index]['val_accuracy']); print('Best epoch:', training_metrics[best_index]['epoch']); print('Total recorded epochs:', len(training_metrics)); print('Final training accuracy:', final_row['accuracy'])

## Per-class performance and error analysis
Low-recall and low-F1 classes are evidence-based candidates for further investigation. The confusion matrix shows which labels are most frequently confused; no misclassified image examples were saved by the current evaluator.

In [ ]:
per_class = metrics.get('per_class', {})
per_class_rows = [{'Class': name, **values} for name, values in per_class.items()]
display(per_class_rows)
if per_class_rows:
    print('Lowest recall:', min(per_class_rows, key=lambda r: r['recall']))
    print('Lowest F1:', min(per_class_rows, key=lambda r: r['f1']))

## Generalization, efficiency, and critical discussion
Compare the final training, validation, and test values cautiously. A gap between training and validation/test suggests some generalization error, while a close validation/test result supports consistency. PlantVillage images are controlled and may not represent field lighting, backgrounds, camera devices, or disease severity. ResNet50 provides strong transfer-learning capacity and residual gradient flow, but its parameter count and CPU training time are substantial. It should not be called the best group model until the other architectures are evaluated under the same protocol.

In [ ]:
print('Total parameters:', metrics.get('total_parameters')); print('Trainable parameters:', metrics.get('trainable_parameters')); print('Checkpoint bytes:', metrics.get('checkpoint_size_bytes')); print('Inference seconds/image:', metrics.get('inference_seconds_per_image')); print('Throughput:', metrics.get('throughput_images_per_second'))
print('Training information:', json.loads((RESULTS_DIR/'training_summary.json').read_text()))

## Future improvements and final summary
Potential improvements include class-balanced sampling or loss, targeted augmentation for minority classes, controlled fine-tuning depth, and external field-image validation. These are future experiments, not claims about the current result.

In [ ]:
summary_table = [{'Model':'ResNet50','Dataset':'PlantVillage','Input size':'224 × 224 RGB','Classes':len(class_names),'Test accuracy':metrics.get('test_accuracy'),'Macro F1':metrics.get('test_f1_macro'),'Weighted F1':metrics.get('test_f1_weighted'),'ROC-AUC':metrics.get('test_roc_auc_ovr_weighted'),'Parameters':metrics.get('total_parameters'),'Training time (s)':json.loads((RESULTS_DIR/'training_summary.json').read_text()).get('total_training_duration_seconds'),'Inference (s/image)':metrics.get('inference_seconds_per_image')} ]
display(summary_table)

## Conclusion and reproducibility
ResNet50 was trained with ImageNet transfer learning and evaluated using multiple classification metrics on an untouched test split. The result is suitable for comparison with the group’s Custom CNN, VGG16, and EfficientNetB0 only when those models use the same classes, split, seed, and evaluation protocol.

Repository branch: `IT23223530_AHAMED_IMT` | Seed: `42` | Results: `results/resnet50/` | Dataset remains external and unchanged. Checkpoints are excluded from Git.